# MetaVR environments, viral taxonomy and recorded host taxonomy

This report describes MetaVR relatives of the shared filtered MOSAIC catalogue. It does not change the catalogue or infer new hosts. Both all-vOTU and Complete/High-quality query subsets are reported. Sample panels describe the assembly origin of catalogue representatives, not presence or abundance inferred from read mapping.


In [ ]:
from pathlib import Path
import base64
import hashlib
import html
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from Bio import SeqIO
from IPython.display import display

input_fasta = Path(snakemake.input.fasta)
input_blast = Path(snakemake.input.blast)
input_checkv = Path(snakemake.input.checkv)
input_uvig_metadata = Path(snakemake.input.uvig_metadata)
input_source_metadata = Path(snakemake.input.source_metadata)
samples = list(dict.fromkeys(snakemake.params.samples))
chunksize = int(snakemake.params.metadata_chunksize)
if chunksize < 1:
    raise ValueError("metadata_chunksize must be positive")
for output_path in snakemake.output:
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
coverage_labels = ["<10%", "10–<50%", "50–<85%", "85–100%"]
subset_names = ["All vOTUs", "Complete/High-quality"]
missing_values = ["\\N", "", "NA", "N/A", "None"]
warnings_report = []

def read_metadata(path, **kwargs):
    return pd.read_csv(path, sep="\t", dtype=str, na_values=missing_values, **kwargs)

def write_table(frame, path):
    frame.to_csv(path, sep="\t", index=False)

def file_signature(path):
    info = path.stat()
    return {"path": str(path.resolve()), "size": info.st_size, "mtime_ns": info.st_mtime_ns}


## 1. Query catalogue and quality subsets

The Complete/High-quality subset uses the query representative's CheckV quality, not the quality of its MetaVR hit. Unassigned-origin contigs, unknown quality, and queries with no hits remain visible. Sample attribution uses the longest matching configured sample prefix; it is not a sample-presence analysis.


In [ ]:
with open(input_fasta) as handle:
    records = list(SeqIO.parse(handle, "fasta"))
ids = [record.id for record in records]
if len(ids) != len(set(ids)):
    raise ValueError("The query FASTA contains duplicate identifiers")

def origin_sample(identifier):
    for sample in sorted(samples, key=lambda name: (-len(name), name)):
        if identifier == sample or identifier.startswith(sample + "_") or identifier.startswith("nanopore_" + sample + "_"):
            return sample
    return "Unassigned origin"

queries = pd.DataFrame({
    "votu_id": pd.Series(ids, dtype=str), "length_bp": [len(record.seq) for record in records],
    "origin_sample": pd.Series([origin_sample(identifier) for identifier in ids], dtype=str),
})
checkv_columns = ["contig_id", "contig_length", "provirus", "proviral_length",
                  "gene_count", "viral_genes", "host_genes", "checkv_quality",
                  "miuvig_quality", "completeness", "completeness_method",
                  "contamination", "kmer_freq", "warnings"]
quality = read_metadata(input_checkv, names=checkv_columns, header=None)
quality = quality.loc[quality["contig_id"].isin(ids), ["contig_id", "checkv_quality", "completeness"]]
if quality.groupby("contig_id")["checkv_quality"].nunique(dropna=False).gt(1).any():
    raise ValueError("Conflicting CheckV quality assignments for the same query")
quality = quality.drop_duplicates("contig_id").rename(columns={
    "contig_id": "votu_id", "checkv_quality": "query_checkv_quality", "completeness": "query_completeness",
})
queries = queries.merge(quality, on="votu_id", how="left", validate="one_to_one")
queries["query_checkv_quality"] = queries["query_checkv_quality"].fillna("Not-determined")
queries["is_complete_or_high_quality"] = queries["query_checkv_quality"].isin(["Complete", "High-quality"])
sample_order = samples + (["Unassigned origin"] if queries["origin_sample"].eq("Unassigned origin").any() else [])
print(f"Query vOTUs: {len(queries):,}; Complete/High-quality: {queries['is_complete_or_high_quality'].sum():,}")
display(queries.head(10))


## 2. Unique query–subject pairs

Reuse the existing 11-column BLAST table (headerless or with its exact header). HSPs are aggregated once per query–subject pair. Query coverage is qcovs, not subject coverage. HSP-weighted identity is descriptive: overlapping HSPs are not resolved into ANI, and these matches are not new species-level assignments.


In [ ]:
blast_columns = ["qseqid", "sseqid", "salltitles", "qstart", "qend", "qlen", "slen", "qcovs", "evalue", "length", "pident"]
pair_columns = ["qseqid", "sseqid", "subject_title", "n_hsps", "qcovs", "qcovs_max",
                "qlen", "qlen_max", "slen", "slen_max", "best_hsp_pident", "best_evalue",
                "summed_hsp_bases", "identity_weight"]
with open(input_blast) as handle:
    first_line = handle.readline().rstrip("\r\n")
has_header = first_line.split("\t") == blast_columns
if first_line and len(first_line.split("\t")) != len(blast_columns):
    raise ValueError("Expected the workflow's 11-column MetaVR BLAST table")
parts = []
if first_line:
    for chunk in pd.read_csv(input_blast, sep="\t", names=blast_columns, header=None,
                             skiprows=int(has_header), dtype=str, keep_default_na=False, chunksize=chunksize):
        chunk = chunk.loc[chunk["qseqid"].isin(ids)].copy()
        if chunk.empty:
            continue
        for column in blast_columns[3:]:
            chunk[column] = pd.to_numeric(chunk[column], errors="raise")
        if (chunk[blast_columns[3:]].isna().any().any() or chunk["sseqid"].eq("").any()
                or not chunk["qcovs"].between(0, 100).all() or not chunk["pident"].between(0, 100).all()
                or chunk[["qlen", "slen", "length"]].le(0).any().any()):
            raise ValueError("Invalid numeric values or subject IDs in MetaVR BLAST")
        chunk["identity_weight"] = chunk["length"] * chunk["pident"]
        parts.append(chunk.groupby(["qseqid", "sseqid"], as_index=False).agg(
            subject_title=("salltitles", "first"), n_hsps=("sseqid", "size"),
            qcovs=("qcovs", "min"), qcovs_max=("qcovs", "max"),
            qlen=("qlen", "min"), qlen_max=("qlen", "max"),
            slen=("slen", "min"), slen_max=("slen", "max"),
            best_hsp_pident=("pident", "max"), best_evalue=("evalue", "min"),
            summed_hsp_bases=("length", "sum"), identity_weight=("identity_weight", "sum"),
        ))
if parts:
    pairs = pd.concat(parts, ignore_index=True).groupby(["qseqid", "sseqid"], as_index=False).agg({
        "subject_title": "first", "n_hsps": "sum", "qcovs": "min", "qcovs_max": "max",
        "qlen": "min", "qlen_max": "max", "slen": "min", "slen_max": "max",
        "best_hsp_pident": "max", "best_evalue": "min", "summed_hsp_bases": "sum", "identity_weight": "sum",
    })
else:
    pairs = pd.DataFrame(columns=pair_columns)
for column in ["qcovs", "qlen", "slen"]:
    if not pairs[column].eq(pairs[column + "_max"]).all():
        raise ValueError(f"Inconsistent {column} within a query–subject pair")
if not pairs["qlen"].eq(pairs["qseqid"].map(queries.set_index("votu_id")["length_bp"])).all():
    raise ValueError("BLAST query lengths do not match the current catalogue; check for stale results")
pairs["hsp_weighted_pident"] = pd.to_numeric(pairs["identity_weight"]) / pd.to_numeric(pairs["summed_hsp_bases"])
pairs = pairs.drop(columns=["qcovs_max", "qlen_max", "slen_max", "identity_weight"])
pairs["qcovs"] = pd.to_numeric(pairs["qcovs"])
pairs["uvig"] = pairs["sseqid"].str.split("|", regex=False).str[0]
pairs["hit_source_id"] = pairs["sseqid"].str.extract(r"^[^|]+\|([^|]+)\|", expand=False)
pairs["coverage_tier"] = pd.cut(pairs["qcovs"], [0, 10, 50, 85, 100.000001],
                                labels=coverage_labels, right=False, include_lowest=True)
print(f"Unique query–subject pairs: {len(pairs):,}; unique MetaVR UViGs: {pairs['uvig'].nunique():,}")


## 3. MetaVR metadata and provenance

Read metadata in chunks and retain only matching UViGs/source datasets. Recorded host lineage and assignment method are kept together; no new iPHoP runs or transfer of a relative's host to the query is performed. Missing metadata and unassigned ranks stay explicit. Cache reuse requires matching requested IDs, columns and source-file signatures.


In [ ]:
uvig_columns = ["uvig", "taxon_oid", "quality", "completeness", "genome_type",
                "ictv_taxonomy", "ictv_taxonomy_method", "host_taxonomy", "host_taxonomy_method"]
source_columns = ["IMG Genome ID", "Genome Name / Sample Name", "GOLD Analysis Project Type",
                  "Ecosystem", "Ecosystem Category", "Ecosystem Type", "Ecosystem Subtype",
                  "Specific Ecosystem", "Geographic Location", "Isolation Country"]
targets = set(pairs["uvig"])
signature = {
    "schema_version": 1, "uvig_columns": uvig_columns, "source_columns": source_columns,
    "uvig_ids_sha256": hashlib.sha256("\n".join(sorted(targets)).encode()).hexdigest(),
    "uvig_source": file_signature(input_uvig_metadata), "dataset_source": file_signature(input_source_metadata),
}
cache = Path(snakemake.output.metadata_subset)
manifest = Path(snakemake.output.metadata_manifest)
try:
    cached_signature = json.loads(manifest.read_text())
except (FileNotFoundError, json.JSONDecodeError):
    cached_signature = None

def select_metadata(path, columns, key, wanted):
    available = read_metadata(path, nrows=0).columns
    missing = set(columns) - set(available)
    if missing:
        raise ValueError(f"Missing metadata columns in {path}: {sorted(missing)}")
    selected = []
    if wanted:
        for chunk in read_metadata(path, usecols=columns, chunksize=chunksize):
            rows = chunk.loc[chunk[key].isin(wanted)]
            if not rows.empty:
                selected.append(rows)
    result = pd.concat(selected, ignore_index=True) if selected else pd.DataFrame(columns=columns)
    if result[key].duplicated().any():
        raise ValueError(f"Duplicate selected {key} values in {path}")
    return result

if cache.exists() and cached_signature == signature:
    metadata = read_metadata(cache)
    required = set(uvig_columns + source_columns + ["metavr_metadata_found", "source_metadata_found"])
    if not required.issubset(metadata.columns) or metadata["uvig"].duplicated().any() or set(metadata["uvig"]) != targets:
        raise ValueError("Invalid MetaVR metadata cache; remove the report cache and rerun")
    for column in ["metavr_metadata_found", "source_metadata_found"]:
        metadata[column] = metadata[column].map({"True": True, "False": False})
        if metadata[column].isna().any():
            raise ValueError(f"Invalid {column} in MetaVR metadata cache")
    print("Reusing selected MetaVR metadata")
else:
    selected = select_metadata(input_uvig_metadata, uvig_columns, "uvig", targets)
    selected["metavr_metadata_found"] = True
    metadata = pd.DataFrame({"uvig": sorted(targets)}, dtype=str).merge(selected, on="uvig", how="left", validate="one_to_one")
    metadata["metavr_metadata_found"] = metadata["metavr_metadata_found"].eq(True)
    # Join physical sources by the database's taxon_oid, not a guessed host identity.
    metadata["IMG Genome ID"] = metadata["taxon_oid"]
    sources = select_metadata(input_source_metadata, source_columns, "IMG Genome ID", set(metadata["taxon_oid"].dropna()))
    sources["source_metadata_found"] = True
    metadata = metadata.merge(sources, on="IMG Genome ID", how="left", validate="many_to_one")
    metadata["source_metadata_found"] = metadata["source_metadata_found"].eq(True)
    write_table(metadata, cache)
    manifest.write_text(json.dumps(signature, indent=2) + "\n")

missing_uvigs = int((~metadata["metavr_metadata_found"]).sum())
missing_sources = int((~metadata["source_metadata_found"]).sum())
if missing_uvigs:
    warnings_report.append(f"{missing_uvigs} hit UViGs lack MetaVR metadata; their hits remain in the report.")
if missing_sources:
    warnings_report.append(f"{missing_sources} hit UViGs lack source-dataset metadata.")
viral_prefixes = {"realm": "r", "kingdom": "k", "phylum": "p", "class": "c", "order": "o", "family": "f", "genus": "g", "species": "s"}
host_prefixes = {"domain": "d", "kingdom": "k", "phylum": "p", "class": "c", "order": "o", "family": "f", "genus": "g", "species": "s"}
for kind, lineage, prefixes in [("viral", "ictv_taxonomy", viral_prefixes), ("host", "host_taxonomy", host_prefixes)]:
    for rank, prefix in prefixes.items():
        metadata[kind + "_" + rank] = metadata[lineage].str.extract(
            rf"(?:^|;)\s*{prefix}__([^;]*)", expand=False,
        ).str.strip().replace("", pd.NA).fillna("Unassigned")
metadata["host_assignment_method"] = metadata["host_taxonomy_method"].fillna("Method not reported")
metadata["host_lineage_recorded"] = metadata["host_taxonomy"].notna()


## 4. Physical source environments

Keep the original GOLD hierarchy and geographic fields in the tables. These categories describe where the MetaVR relatives were recovered; they are not evidence that the query infects that environment's host. Marine/coastal and freshwater sources remain distinct.


In [ ]:
hit_data = pairs.merge(metadata, on="uvig", how="left", validate="many_to_one")
hit_data = hit_data.merge(queries, left_on="qseqid", right_on="votu_id", how="left", validate="many_to_one")
hierarchy = ["Ecosystem", "Ecosystem Category", "Ecosystem Type", "Ecosystem Subtype", "Specific Ecosystem"]
source_text = hit_data[hierarchy].fillna("").agg(" | ".join, axis=1).str.lower() if len(hit_data) else pd.Series(index=hit_data.index, dtype=str)
category = hit_data["Ecosystem Category"].fillna("")
ecosystem = hit_data["Ecosystem"].fillna("")
gut = source_text.str.contains(r"digestive|intestin|fecal|faecal|\bgut\b|rumen", regex=True)
environment_order = ["Human feces/gut", "Other animal gut", "Wastewater/WWTP", "Marine/coastal",
                     "Freshwater", "Other aquatic", "Soil/terrestrial", "Other host-associated",
                     "Other engineered", "Isolate; source not reported", "Other/unspecified", "Environment not reported"]
conditions = [
    category.eq("Mammals: Human") & gut, ~category.eq("Mammals: Human") & gut,
    source_text.str.contains(r"wastewater|sewage|wwtp|activated sludge"),
    source_text.str.contains(r"\bmarine\b|ocean|seawater|coastal|intertidal|estuar|harbor|strait|salt marsh"),
    source_text.str.contains(r"freshwater|\briver\b|\blake\b|\bpond\b|\bcreek\b|\blotic\b|wetland"),
    category.eq("Aquatic"), source_text.str.contains(r"terrestrial|\bsoil\b|pasture|agricultural land"),
    ecosystem.str.contains("Host-associated", case=False), ecosystem.eq("Engineered"),
    hit_data["GOLD Analysis Project Type"].fillna("").eq("Genome Analysis (Isolate)"),
    source_text.str.replace("|", "", regex=False).str.strip().ne(""),
]
hit_data["environment_group"] = np.select(conditions, environment_order[:-1], default=environment_order[-1])
write_table(hit_data, snakemake.output.hit_pairs)


## 5. Per-vOTU summaries and explicit sample statuses

No-hit queries remain in the summary. The strongest saved hit is chosen by query coverage, HSP-weighted identity, best HSP identity, E-value, then subject ID. Its host metadata are labelled as hit metadata, not as a query-host assignment.

Each configured sample is retained in both views: No vOTUs in this subset, No MetaVR hits, No hits ≥50% coverage, or Hits ≥50% coverage. Deduplication can select another sample's contig as representative, so No vOTUs in this subset does not mean no viruses were present in that sample.


In [ ]:
hit_counts = hit_data.groupby("qseqid", as_index=False).agg(
    n_metavr_hits=("sseqid", "size"), n_hits_qcov50=("qcovs", lambda x: int(x.ge(50).sum())),
    n_hits_qcov85=("qcovs", lambda x: int(x.ge(85).sum())), max_qcovs=("qcovs", "max"),
).rename(columns={"qseqid": "votu_id"})
top_hits = hit_data.sort_values(
    ["qseqid", "qcovs", "hsp_weighted_pident", "best_hsp_pident", "best_evalue", "sseqid"],
    ascending=[True, False, False, False, True, True],
).drop_duplicates("qseqid")
top_columns = ["sseqid", "uvig", "qcovs", "hsp_weighted_pident", "best_hsp_pident", "best_evalue",
               "quality", "completeness", "genome_type", "ictv_taxonomy", "ictv_taxonomy_method",
               "host_taxonomy", "host_taxonomy_method", "host_genus", "environment_group",
               "IMG Genome ID", "Geographic Location", "Isolation Country", "metavr_metadata_found"]
top = top_hits[["qseqid"] + top_columns].rename(columns={
    "qseqid": "votu_id", **{column: "top_hit_" + column for column in top_columns},
})
query_summary = queries.merge(hit_counts, on="votu_id", how="left", validate="one_to_one").merge(top, on="votu_id", how="left", validate="one_to_one")
for column in ["n_metavr_hits", "n_hits_qcov50", "n_hits_qcov85"]:
    query_summary[column] = query_summary[column].fillna(0).astype(int)
write_table(query_summary, snakemake.output.query_summary)
sample_rows, run_rows, environment_frames, viral_frames, host_frames, method_frames = [], [], [], [], [], []

def taxonomy_counts(frame, prefixes, kind, subset):
    frames = []
    for rank in prefixes:
        column = kind + "_" + rank
        unique = frame.drop_duplicates("uvig")[column].value_counts().rename_axis("taxon").reset_index(name="n")
        unique["counting_unit"], unique["coverage_tier"] = "unique_subject", "All hits"
        pair = frame.groupby(["coverage_tier", column], observed=True).size().reset_index(name="n").rename(columns={column: "taxon"})
        pair["counting_unit"] = "query_subject_pair"
        both = pd.concat([unique, pair], ignore_index=True)
        both["rank"], both["subset"] = rank, subset
        frames.append(both)
    return pd.concat(frames, ignore_index=True)[["subset", "counting_unit", "coverage_tier", "rank", "taxon", "n"]]

for subset in subset_names:
    q = query_summary if subset == subset_names[0] else query_summary.loc[query_summary["is_complete_or_high_quality"]]
    hits = hit_data.loc[hit_data["qseqid"].isin(q["votu_id"])]
    for sample in sample_order:
        sq = q.loc[q["origin_sample"].eq(sample)]
        strong = int(sq["n_hits_qcov50"].sum())
        total = int(sq["n_metavr_hits"].sum())
        status = ("No vOTUs in this subset" if sq.empty else "No MetaVR hits" if total == 0
                  else "No hits ≥50% coverage" if strong == 0 else "Hits ≥50% coverage")
        sample_rows.append([subset, sample, len(sq), int(sq["n_metavr_hits"].gt(0).sum()),
                            total, strong, status])
    run_rows.append([subset, len(q), int(q["n_metavr_hits"].eq(0).sum()), len(hits),
                     hits["uvig"].nunique(), int(q["n_hits_qcov50"].gt(0).sum()),
                     hits.loc[~hits["metavr_metadata_found"].eq(True), "uvig"].nunique()])
    ec = hits.groupby(["coverage_tier", "environment_group"], observed=True).size().reset_index(name="n_pairs")
    ec.insert(0, "subset", subset)
    environment_frames.append(ec)
    viral_frames.append(taxonomy_counts(hits, viral_prefixes, "viral", subset))
    host_frames.append(taxonomy_counts(hits, host_prefixes, "host", subset))
    methods = hits.drop_duplicates("uvig").groupby(["host_assignment_method", "host_lineage_recorded"]).size().reset_index(name="n_unique_subjects")
    methods.insert(0, "subset", subset)
    method_frames.append(methods)
sample_summary = pd.DataFrame(sample_rows, columns=["subset", "origin_sample", "n_votus", "n_votus_with_hits", "n_hit_pairs", "n_hit_pairs_qcov50", "status"])
run_summary = pd.DataFrame(run_rows, columns=["subset", "n_votus", "n_votus_without_hits", "n_hit_pairs", "n_unique_subjects", "n_votus_with_qcov50_hits", "n_subjects_missing_metadata"])
environment_counts = pd.concat(environment_frames, ignore_index=True)
viral_counts = pd.concat(viral_frames, ignore_index=True)
host_counts = pd.concat(host_frames, ignore_index=True)
method_counts = pd.concat(method_frames, ignore_index=True)
for frame, output in [(sample_summary, snakemake.output.sample_summary), (run_summary, snakemake.output.run_summary),
                      (environment_counts, snakemake.output.environment_counts), (viral_counts, snakemake.output.viral_taxonomy_counts),
                      (host_counts, snakemake.output.host_taxonomy_counts), (method_counts, snakemake.output.host_method_counts)]:
    write_table(frame, output)
display(run_summary)
display(sample_summary)


## 6. Coverage-stratified and sample-origin composition

Bars count unique query–subject pairs, not independent samples or read abundance. Host genera and viral families outside the 12 most frequent assigned categories are pooled only in figures; exported tables keep all ranks and taxa. Grey indicates unassigned taxonomy. Zero-hit samples are labelled rather than given fabricated percentage bars.


In [ ]:
def plot_categories(column, fixed=None):
    if fixed is not None:
        return fixed
    counts = hit_data[column].value_counts()
    assigned = [name for name in counts.index if name != "Unassigned"]
    return assigned[:12] + (["Other assigned taxa"] if len(assigned) > 12 else []) + ["Unassigned"]

category_orders = {
    "environment_group": plot_categories("environment_group", environment_order),
    "viral_family": plot_categories("viral_family"),
    "host_genus": plot_categories("host_genus"),
}
palette = plt.get_cmap("tab20")
category_colors = {
    column: {name: ("#B8B8B8" if name in ["Unassigned", "Environment not reported"] else palette(i % 20))
             for i, name in enumerate(order)}
    for column, order in category_orders.items()
}

def composition(ax, hits, group_column, groups, category_column, title, statuses=None):
    frame = hits.copy()
    categories = category_orders[category_column]
    frame["plot_category"] = frame[category_column].where(frame[category_column].isin(categories), "Other assigned taxa")
    counts = frame.groupby([group_column, "plot_category"], observed=True).size().unstack(fill_value=0)
    counts = counts.reindex(index=groups, columns=categories, fill_value=0).fillna(0)
    totals = counts.sum(axis=1)
    percentages = counts.div(totals.replace(0, np.nan), axis=0).mul(100).fillna(0)
    y = np.arange(len(groups))
    left = np.zeros(len(groups))
    for category in categories:
        values = percentages[category].to_numpy(dtype=float)
        if values.sum() > 0:
            ax.barh(y, values, left=left, color=category_colors[category_column][category], label=category)
        left += values
    ax.set_yticks(y, [f"{group} (n={int(totals.loc[group])})" for group in groups])
    for i, group in enumerate(groups):
        if totals.loc[group] == 0:
            ax.text(2, i, statuses.get(group, "No qualifying hits") if statuses is not None else "No hits in this tier",
                    va="center", fontsize=9, color="#555555", bbox=dict(facecolor="white", edgecolor="none", pad=1))
    if not groups:
        ax.text(0.5, 0.5, "No query samples", transform=ax.transAxes, ha="center")
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Query–subject pairs (%)")
    ax.set_title(title, loc="left")
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, -0.22),
                  fontsize=7, ncol=3, frameon=False)
    ax.grid(axis="x", alpha=0.3)
    ax.grid(False, axis="y")
    ax.set_axisbelow(True)

fig, axes = plt.subplots(3, 2, figsize=(18, 16), constrained_layout=True)
for col, subset in enumerate(subset_names):
    hits = hit_data if col == 0 else hit_data.loc[hit_data["is_complete_or_high_quality"]]
    for row, (column, label) in enumerate([("environment_group", "Source environments"),
                                          ("viral_family", "MetaVR viral families"), ("host_genus", "Recorded host genera")]):
        composition(axes[row, col], hits, "coverage_tier", coverage_labels, column, f"{subset}: {label}")
fig.savefig(snakemake.output.composition_png, dpi=150, bbox_inches="tight")
fig.savefig(snakemake.output.composition_svg, bbox_inches="tight")
display(fig)
plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(20, max(12, 0.75 * len(sample_order) + 6)), constrained_layout=True)
for col, subset in enumerate(subset_names):
    hits = hit_data if col == 0 else hit_data.loc[hit_data["is_complete_or_high_quality"]]
    hits = hits.loc[hits["qcovs"].ge(50)]
    statuses = sample_summary.loc[sample_summary["subset"].eq(subset)].set_index("origin_sample")["status"].to_dict()
    for row, (column, label) in enumerate([("environment_group", "Source environments"), ("host_genus", "Recorded host genera")]):
        composition(axes[row, col], hits, "origin_sample", sample_order, column,
                    f"{subset}: {label}, query coverage ≥50%", statuses)
fig.suptitle("Catalogue representative assembly origin — not sample presence or read abundance")
fig.savefig(snakemake.output.samples_png, dpi=150, bbox_inches="tight")
fig.savefig(snakemake.output.samples_svg, bbox_inches="tight")
display(fig)
plt.close(fig)


## 7. Export report and analysis provenance

The report preserves unknown hosts, missing metadata, and no-hit samples. Host taxonomy belongs to the matched MetaVR sequences; physical source, host identity and query-host inference are different concepts. No iPHoP, TaxMyPhage or VIRIDIC analysis is launched here. The existing saved BLAST search limits which relatives can be described.


In [ ]:
provenance = {
    "inputs": {name: file_signature(Path(path)) for name, path in snakemake.input.items()},
    "sampling": str(snakemake.params.sampling), "samples": samples,
    "query_subset_quality_field": "CheckV checkv_quality", "hq_categories": ["Complete", "High-quality"],
    "coverage_tiers": coverage_labels, "sample_plot_min_qcovs": 50,
    "host_source": "MetaVR recorded host_taxonomy and host_taxonomy_method; no new predictions",
    "sample_interpretation": "assembly origin of representative; not detection or abundance",
    "warnings": warnings_report,
}
Path(snakemake.output.provenance).write_text(json.dumps(provenance, indent=2) + "\n")
def embedded_image(path, alt):
    data = base64.b64encode(Path(path).read_bytes()).decode("ascii")
    return f'<img alt="{html.escape(alt)}" src="data:image/png;base64,{data}" style="max-width:100%">'

report_path = Path(snakemake.output.summary_html)
links = []
for name, path in snakemake.output.items():
    if name != "summary_html":
        relative = Path(path).relative_to(report_path.parent)
        links.append(f'<li><a href="{html.escape(str(relative), quote=True)}">{html.escape(name)}</a></li>')
sections = [
    "<!doctype html><html><head><meta charset='utf-8'><title>MOSAIC MetaVR analysis</title>"
    "<style>body{font-family:sans-serif;margin:2em}table{border-collapse:collapse;font-size:0.9em}"
    "td,th{padding:0.35em;border:1px solid #ddd}th{background:#eee}</style></head><body>",
    "<h1>MOSAIC MetaVR analysis</h1>",
    "<p>All filtered vOTUs and the Complete/High-quality subset of query representatives. "
    "Counts describe saved MetaVR matches, not abundance or independent environmental samples. "
    "Host taxonomy belongs to the MetaVR relatives and is not a demonstrated host of the query. "
    "HSP-weighted identity is descriptive, not ANI.</p>",
    "<h2>Overview</h2>", run_summary.to_html(index=False, escape=True),
    "<h2>Representative assembly origin</h2><p>All configured samples are retained. "
    "No vOTUs in this subset does not imply absence of viruses: representatives may originate from another sample.</p>",
    sample_summary.to_html(index=False, escape=True),
    "<h2>Recorded host assignment methods</h2>", method_counts.to_html(index=False, escape=True),
    "<h2>Coverage composition</h2>", embedded_image(snakemake.output.composition_png, "Coverage composition"),
    "<h2>Sample-origin composition</h2>", embedded_image(snakemake.output.samples_png, "Sample-origin composition"),
    "<h2>Data checks</h2><ul>" + "".join("<li>" + html.escape(message) + "</li>" for message in warnings_report) + "</ul>",
    "<h2>Tables, figures and provenance</h2><ul>" + "".join(links) + "</ul></body></html>",
]
report_path.write_text("\n".join(sections))
print(f"Report: {report_path}")
